# drugMR: Interpreting Results

> *A guided walkthrough of a completed **drugMR** run outside the dashboard, dissecting the results tables directly, from the first genetic signal to the final evidence profile of each prioritised target.*

### Overview

Rather than browsing results in the dashboard, this notebook opens the raw results tables of a completed run and dissects them one stage at a time. Each section loads the relevant table, shows what its columns mean, and highlights what separates a convincing target from a weak one.

### Requirements

| | |
|:--|:--|
| **Repository** | The `drugMR` repository cloned locally. |
| **Environment** | Created with `env/bootstrap.sh`, which installs the `drugmr` package in editable mode. No separate `pip install` is needed. |
| **Kernel** | This notebook running on the environment's kernel (`.venv` in the repository root). |
| **Run** | A completed run under `runs/<run_id>/`. If the pipeline ran on a remote HPC, fetch it first with `notebooks/00_drugmr.ipynb`. |

In [ ]:
from drugmr.interpret import ResultsExplorer

In [ ]:
explorer = ResultsExplorer(
    run_id="AD_wu_csf_2026-10-09_18-51-14_1989e4d",
    root="../runs",
)

## 1. Run overview

What the run is, the thresholds every later stage is judged against, and how many proteins survive each stage.

**`run_info()`** lists the run identity, the code and container that produced it, the outcome GWAS inputs and every gate.

In [ ]:
explorer.run_info()

**`funnel()`** counts the proteins remaining after each stage. The share column is taken against the stage named in the note.

In [ ]:
explorer.funnel()

## 2. Causal estimates (cis-MR)

The effect of each protein on the outcome. When a higher protein level raises risk, the drug direction is to inhibit; when it lowers risk, the drug direction is to activate.

**`cis_mr()`** gives one row per protein passing the cis-MR gate, with the odds ratio and 95% CI, P, FDR and drug direction.

In [ ]:
explorer.cis_mr()

With `hits_only=False` it returns every protein tested, with a column saying whether it passed.

In [ ]:
explorer.cis_mr(hits_only=False)

**`instruments()`** shows the SNPs behind each estimate: alleles, pQTL and outcome effects, F statistic and the Steiger direction test.

In [ ]:
explorer.instruments()

**`sensitivity()`** compares IVW, MR Egger and weighted median for proteins with two or more instruments. Single instrument proteins use a Wald ratio, which has no sensitivity methods.

In [ ]:
explorer.sensitivity()

**`forest_plot()`** plots the odds ratio and 95% CI of every prioritised target.

In [ ]:
explorer.forest_plot();

## 3. Shared causal variant

A causal estimate is only convincing when the protein and the outcome share the same causal variant. COLOC assumes one signal per trait; PWCoCo conditions on each independent signal, so it can recover windows holding more than one.

**`colocalisation()`** gives COLOC and PWCoCo for each prioritised target, the best PWCoCo pair and which method supports it.

In [ ]:
explorer.colocalisation()

With `hits_only=False` it covers every cis-MR hit, including those that did not colocalise.

In [ ]:
explorer.colocalisation(hits_only=False)

**`coloc_priors()`** repeats COLOC under stricter priors. A target whose PP4 stays above the threshold under all of them is robust to the choice of prior.

In [ ]:
explorer.coloc_priors()

**`lead_variants()`** shows the top cis variant of each protein with its outcome and pQTL effects on the same allele, and whether the direction they imply matches cis-MR.

In [ ]:
explorer.lead_variants()

## 4. xQTL triangulation

Whether QTL for other molecular traits of the same gene (xQTL) point to the same signal and direction as the pQTL. Bulk tissue and single cell xQTL are reported separately, and each check states whether its direction agrees with cis-MR.

**`smr()`** lists significant SMR results with a passing HEIDI test, one row per protein and tissue or cell type.

In [ ]:
explorer.smr()

The `data_type` argument restricts it to bulk tissue or to single cell xQTL.

In [ ]:
explorer.smr(data_type="bulk")

In [ ]:
explorer.smr(data_type="single_cell")

**`hyprcoloc()`** lists clusters in which the pQTL, the outcome GWAS and an xQTL share one causal variant (posterior of at least 0.7).

In [ ]:
explorer.hyprcoloc()

With `supported_only=False` it returns every cluster, including those missing a trait or below the posterior threshold.

In [ ]:
explorer.hyprcoloc(supported_only=False)

**`pwcoco_qtl()`** gives PWCoCo of each xQTL against the pQTL and against the outcome GWAS.

In [ ]:
explorer.pwcoco_qtl()

**`directions()`** brings the direction checks together, one row per target. A target is consistent when no check opposes cis-MR and mixed when at least one does.

In [ ]:
explorer.directions()

## 5. Safety and repurposing

Other traits affected by the same instruments. A trait moving in the same direction as the outcome is a possible additional indication; one moving in the opposite direction is a possible adverse effect.

**`phewas()`** lists the Bonferroni significant traits of each protein, from FinnGen and then UK Biobank, with their class.

In [ ]:
explorer.phewas()

## 6. Evidence summary

**`targets()`** gives one row per prioritised target, summarising every stage above.

In [ ]:
explorer.targets()

## 7. One target end to end: TREM2

Every function above also accepts a protein, by symbol in any case or by full protein ID, for example `explorer.smr("TREM2")`.

**`target()`** runs every function for one protein in reading order, and returns the tables as a dictionary.

In [ ]:
trem2 = explorer.target("TREM2")

**`regional_plot()`** shows the outcome GWAS and the pQTL across the cis window, coloured by LD with the lead variant.

In [ ]:
explorer.regional_plot("TREM2");